In [17]:
import pandas as pd

RANDOM_STATE = 42

In [18]:
df = pd.read_csv('tweets.csv').drop('Unnamed: 0', axis=1)
df.head()

,username,text,is_quote_status,quote_count,reply_count,favorite_count,view_count,retweet_count,has_card,thumbnail_title,...,ticker,day,month,year,is_in_reply_to,is_view_count,open_last,close_1_day_after,close_3_day_after,close_7_day_after
0,davidfaber,The chance of $MSFT winning an appeal of the ...,0,1,43,135,113387.0,19,0,NaN,...,MSFT,26,4,2023,0,1,296.700,304.83,305.56,304.40
1,PhilipEtienne,We love and appreciate all the volunteers at t...,0,0,0,3,707.0,1,0,NaN,...,NVDA,6,5,2024,0,1,893.900,905.54,887.47,903.99
2,PhilipEtienne,Today walking the lab on the NJ beach - tomorr...,0,0,0,2,702.0,0,0,NaN,...,META,8,5,2024,0,1,463.500,475.42,468.01,481.54
3,PhilipEtienne,Today walking the lab on the NJ beach - tomorr...,0,0,0,2,702.0,0,0,NaN,...,NVDA,8,5,2024,0,1,894.825,887.47,903.99,946.30
4,PhilipEtienne,Good Morning from a dog walk on the $NVDA resc...,0,0,2,7,934.0,0,0,NaN,...,NVDA,5,5,2024,0,1,877.890,921.40,921.40,898.78


In [19]:
df.isna().sum()

username               0
text                   0
is_quote_status        0
quote_count            0
reply_count            0
favorite_count         0
view_count             0
retweet_count          0
has_card               0
thumbnail_title      448
urls                   0
hashtags             505
ticker                 0
day                    0
month                  0
year                   0
is_in_reply_to         0
is_view_count          0
open_last              0
close_1_day_after      0
close_3_day_after      0
close_7_day_after      0
dtype: int64

In [20]:
df = df.fillna('')

### Описание атрибутов ###
- **username** - Username
- **name** - Author of the tweet
- **text** - The full text of the tweet
- **lang** - The language of the tweet
- **in_reply_to** - The tweet ID this tweet is in reply to, if any
- **is_quote_status** - Indicates if the tweet is a quote status
- **retweeted_tweet** - The Tweet being retweeted (if any)
- **possibly_sensitive** - Indicates if the tweet content may be sensitive
- **favorited** - Indicates if the tweet is favorited
- **date** - The date and time when the tweet was created
- **quote_count** - The count of quotes for the tweet
- **reply_count** - The count of replies to the tweet
- **favorite_count** - The count of favorites or likes for the tweet
- **view_count** - The count of views
- **view_count_state** - The state of the tweet views
- **retweet_count** - The count of retweets for the tweet
- **place** - The location associated with the tweet
- **is_translatable** - Indicates if the tweet is translatable
- **edits_remaining** - The remaining number of edits allowed for the tweet
- **has_card** - Indicates if the tweet contains a card
- **thumbnail_title** - The title of the webpage displayed inside the tweet’s card
- **urls** - Information about URLs contained in the tweet
- **hashtags** - Hashtags included in the tweet text

In [21]:
from sklearn.base import BaseEstimator, TransformerMixin

class СlsTargetTransformer(BaseEstimator, TransformerMixin):
    '''
    Класс трансформера, создающего таргет для задачи классификации.
    Положительная категория задается через процент минимальной прибыли.
    '''
    def __init__(self, shift_percent=0):
        """
        Инициализация трансформера с гиперпараметрами.
        :param shift_percent: Порог положительной категории таргета в процентах (опционально).
        """
        self.shift_percent = shift_percent

    def transform(self, X):
        """
        Метод transform: применяет преобразование к данным.
        :param X: Входные данные.
        :return: Преобразованные данные.
        """
        X_copy = X.copy()
        
        for k in [1, 3, 7]:
            col_name = f'{k}_day_after'
            X_copy[col_name] = (X_copy[f'close_{k}_day_after'] / X_copy['open_last'] - 1) * 100 - self.shift_percent
            X_copy[col_name] = X_copy[col_name].apply(lambda x: 1 if x >= 0 else 0)
             
        X_copy = X_copy.drop(['close_1_day_after', 'close_3_day_after', 'close_7_day_after'], axis=1)
        
        return X_copy

In [24]:
class_target_transformer = СlsTargetTransformer()

X0 = class_target_transformer.transform(df)

X0.head()

,username,text,is_quote_status,quote_count,reply_count,favorite_count,view_count,retweet_count,has_card,thumbnail_title,...,ticker,day,month,year,is_in_reply_to,is_view_count,open_last,1_day_after,3_day_after,7_day_after
0,davidfaber,The chance of $MSFT winning an appeal of the ...,0,1,43,135,113387.0,19,0,,...,MSFT,26,4,2023,0,1,296.700,1,1,1
1,PhilipEtienne,We love and appreciate all the volunteers at t...,0,0,0,3,707.0,1,0,,...,NVDA,6,5,2024,0,1,893.900,1,0,1
2,PhilipEtienne,Today walking the lab on the NJ beach - tomorr...,0,0,0,2,702.0,0,0,,...,META,8,5,2024,0,1,463.500,1,1,1
3,PhilipEtienne,Today walking the lab on the NJ beach - tomorr...,0,0,0,2,702.0,0,0,,...,NVDA,8,5,2024,0,1,894.825,0,1,1
4,PhilipEtienne,Good Morning from a dog walk on the $NVDA resc...,0,0,2,7,934.0,0,0,,...,NVDA,5,5,2024,0,1,877.890,1,1,1
